In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
# general imports
import scipy
import os
import re
import mne
import numpy as np
import yasa
from scipy.signal import hilbert
import sys

# import from custom script in same directory
import shared_processing_functions as spf

# import from different directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from hdf5_files import Artefacts_Detection as ad

## Paths

In [ ]:
# root directory for all data
main_data_path = r"D:\dilon_data\edf_collection\no-high-pass_static"
# annotation directories
sleep_edf_anno = r"D:\dilon_data\annotation_collection\Sleep-EDF"
motorwp4_anno = r"D:\dilon_data\annotation_collection\MOTORWP4"
# output path
output_path = r"D:\dilon_data\mat_collection\motorwp4_no-high-pass_static_no-artefact-removal_rms"

## variables

In [4]:
# regex pattern to extract subject
pattern = re.compile(r'(SC4\d{3}|ST7\d{3}|S\d{2,3}_\d)')
# wake time (s) to save before first sleep and after last sleep
# (30 mins, so 30(s) * 60(s))
wake_time = 1800

# which channels to extract
channels = ["Fpz-Cz", "Pz-Oz", "horizontal", "submental"]

# stage names and corresponding ids
sleep_edf_stage_id = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
    "Movement time": 5,
    "Sleep stage ?": 6
}

motorwp4_stage_id = {'0':0,'1':1,'2':2,'3':3,'5':4}

# target bands to use for psd
target_bands = ['Noise', 'Delta', 'Theta', 'Sigma', 'Gamma']
# # stage to use for psd

# power band frequencies to use
bands_list = [
    [0, 0.5, 'Noise'],
    [0.5, 4, 'Delta'],
    [5, 11, 'Theta'],
    [11, 17, 'Sigma'],
    [35, 45, 'Gamma']
]

# Main

### Functions

#### Find all edf files

In [ ]:
def find_edf(path):
    """ 
    Saves all .edf files to a list.

    Parameters:
        path (str): Path to folder that contain your .edf files.

    Returns:
        raw_files (list): List with all .edf files.
    """
    raw_files = []
    for file in os.listdir(os.path.join(path)):
        # skip hidden files
        if not file.startswith('.'):
            if ".edf" in file:
                raw_files.append(file)
    
    return raw_files

#### Add anotation to raw object

In [ ]:
def add_annotation(anno_file, raw):
    """
    Adds annotation to raw MNE object.
    
    Parameters:
        anno_file (str): path to annotation file.
        raw (obj): MNE object containing raw EEG data.
    """
    # load mat file with annotation
    mat_data = scipy.io.loadmat(anno_file)
    states = mat_data["states"]

    # get values from 2d array
    descriptions = [str(int(s[0])) for s in states]  # state labels as strings
    onsets = [float(s[2]) for s in states]           # onset in seconds
    durations = [float(s[3]) for s in states]        # duration in seconds

    # create annotations object
    annotations = mne.Annotations(onset=onsets, duration=durations, description=descriptions)

    # set annotations
    raw.set_annotations(annotations)

#### crop data to annotation size

In [ ]:
def crop_to_anno(raw):
    """
    Crop raw object to match annotation length.

    Parameters:
        raw (obj): MNE object containing raw EEG data.

    Returns:
        raw_cropped (obj): MNE object containing cropped EEG data.
    """
    segments = []

    # iterate annotations
    for onset, duration in zip(raw.annotations.onset, raw.annotations.duration):
        tmin = onset - raw.first_time
        tmax = min(tmin + duration, raw.times[-1])

        # skip invalid chunks
        if tmin >= tmax:
            continue

        # create signal segments
        seg = raw.copy().crop(tmin=tmin, tmax=tmax)
        segments.append(seg)

    # combine all segments into a complete signal
    raw_cropped = mne.concatenate_raws(segments)
    
    return raw_cropped

#### RMS smoothing

In [ ]:
def rms(signal, window_size):
    """
    Compute the Root Mean Square (rms) of the EMG signal with a 
    sliding window.

    Parameters:
        signal (array): Array with EMG signal.
        window_size (int): size of sliding window.

    Returns:
        rms (array): rms expressed emg signal

    """
    signal_sq = np.square(signal)
    window = np.ones(window_size)/float(window_size)
    
    # calculate rms with convolution
    rms = np.sqrt(np.convolve(signal_sq, window, 'same'))

    return rms

#### Create EMG channel

In [ ]:
def create_emg_channel(mne_obj, high_pass, low_pass, window_size):
    """
    Create a combined EMG signal from 2 electrodes with some pre-processing.

    Parameters:
        mne_obj (obj): MNE object containing EMG channels.
        high_pass (float/int): Value for the low end cutoff (e.g. 0.5 Hz).
        low_pass (float/int): Value for the high end cutoff (e.g. 49 Hz).
        window_size (int): size of sliding window.

    Returns:
        combined_emg (array): averaged EMG signal expressed in rms.
    """
    # 4th order butterworth bandpass filter
    mne_obj.filter(
        l_freq=high_pass,
        h_freq=low_pass,
        method="iir",
        iir_params=dict(order=4, ftype="butter")
    )

    # split the two channels
    left_channel = mne_obj.copy().pick(["E240"]).get_data()
    right_channel = mne_obj.copy().pick(["E243"]).get_data()

    left_channel = np.abs(left_channel)
    right_channel = np.abs(right_channel)

    # express in rms
    left_channel = rms(np.squeeze(left_channel), window_size)
    right_channel = rms(np.squeeze(right_channel), window_size)

    # take average
    combined_emg = (left_channel + right_channel) / 2

    return combined_emg

#### Create EOG channel

In [ ]:
def create_eog_channel(mne_obj, high_pass, low_pass):
    """ 
    Create a bipolar EOG channel with pre-processing.

    Parameters:
        mne_obj (obj): MNE object containing EMG channels.
        high_pass (float/int): Value for the low end cutoff (e.g. 0.5 Hz).
        low_pass (float/int): Value for the high end cutoff (e.g. 49 Hz).

    Returns:
        bipolar_eog (array):
    """
    # 4th order butterworth bandpass filter
    mne_obj.filter(
        l_freq=high_pass,
        h_freq=low_pass,
        method="iir",
        iir_params=dict(order=4, ftype="butter")
    )

    # split the two channels
    left_channel = mne_obj.copy().pick(["E10"]).get_data()
    right_channel = mne_obj.copy().pick(["E54"]).get_data()

    # manually create bipolar signal
    bipolar_eog = left_channel - right_channel

    return bipolar_eog

### Workflow

#### Get list with all edf files

In [13]:
# find all edf files
edf_files = find_edf(main_data_path)

print(f"Amount of files available: {len(edf_files)}")

Amount of files available: 115


In [ ]:
for edf_file_path in edf_files:
    # dictionary to save the different bands with highest 
    # power in specific channel
    channel_bands = {}
    # create directory to save info to
    final_results = {}

    # find subject
    match = pattern.search(edf_file_path)
    if match:
        old_subject = match.group(1)

    # check if subject name is in the right format
    if "_" not in old_subject:
        subject = old_subject[:-1] + "_" + old_subject[-1]
    else:
        subject = old_subject

    # skips subjects that already exists
    if any(subject in file for file in list(os.listdir(output_path))):
        continue
    else:
        print(f'Extracting data from subject: {subject}')
        print('-' * 50)

        # read raw data
        raw = mne.io.read_raw_edf(
            os.path.join(main_data_path, edf_file_path), 
            channels, 
            infer_types=True
        )

        if "SC" in subject or "ST" in subject:
            # set correct sampling frequency
            fs = 100
            # find correct annotation file
            anno_file = next((
                anno for anno in os.listdir(sleep_edf_anno) 
                if old_subject in anno), 
                None
            )

            # extract and annotate raw data
            spf.add_annotation(os.path.join(sleep_edf_anno, anno_file), raw)

        else:
            # set correct sampling frequency
            fs = 250
            # find correct annotation file (different filetype)
            anno_file = next((
                anno for anno in os.listdir(motorwp4_anno) 
                if subject in anno and ".mat" in anno), 
                None
            )
            # annotate raw
            add_annotation(os.path.join(motorwp4_anno, anno_file), raw)

        ### cropping data
        # check if raw data is longer than annotation
        if raw.times[-1] > raw.annotations.duration.sum():
            # crop raw to annotation
            temp_raw = crop_to_anno(raw)
        else:
            temp_raw = raw

        # crop the raw data
        cropped_raw = spf.crop_data(temp_raw, wake_time)

        # load data for additional filters and processing
        cropped_raw.load_data()

        ### create .mat file for the annotation
        # get sleep states from cropped raw and save to .mat file
        if "SC" in subject or "ST" in subject:
            sleep_states = spf.get_stages(cropped_raw, sleep_edf_stage_id)
            spf.create_mat(output_path, subject, "states", sleep_states)
        else:
            sleep_states = spf.get_stages(cropped_raw, motorwp4_stage_id)
            spf.create_mat(output_path, subject, "states", sleep_states)

        # apply bandpass to cropped signal
        cropped_raw.filter(l_freq=0, h_freq=49, picks=["Fpz-Cz", "Pz-Oz"])

        # isolate Sleep-EDF dataset
        if "SC" in subject or "ST" in subject:
            for channel in ['horizontal', 'submental', 'Fpz-Cz', 'Pz-Oz']:
                cropped_raw_c = cropped_raw.copy().pick([channel])
                # rename indistinct channel names
                if channel == "horizontal":
                    # apply eog specific filters
                    cropped_raw_c.filter(
                        l_freq=0.5,
                        h_freq=35,
                        method="iir",
                        iir_params=dict(order=4, ftype="butter"),
                        picks=['horizontal']
                    )
                    cropped_raw_c.rename_channels({"horizontal": "EOG"})
                    channel = "EOG"
                elif channel == "submental":
                    cropped_raw_c.rename_channels({"submental":"EMG"})
                    channel = "EMG"

                cropped_data = cropped_raw_c.get_data()
                spf.create_mat(output_path, subject, channel, cropped_data)
        # MOTORWP4 dataset
        else:
            for channel in ["Fpz-Cz", "Pz-Oz"]:
                cropped_raw_c = cropped_raw.copy().pick([channel])
                cropped_data = cropped_raw_c.get_data()
                spf.create_mat(output_path, subject, channel, cropped_data)
            
            # create EMG channel
            emg_picks = cropped_raw.copy().pick(["E240", "E243"])
            emg_picks.load_data()
            combined_emg = create_emg_channel(emg_picks, 5, 120, fs) 
            spf.create_mat(output_path, subject, "EMG", combined_emg)

            # create EOG channel
            eog_picks = cropped_raw.copy().pick(["E10", "E54"])
            eog_picks.load_data()
            bipolar_eog = create_eog_channel(eog_picks, 0.5, 35)
            spf.create_mat(output_path, subject, "EOG", bipolar_eog)